In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.autograd import Variable
import numpy as np
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets
import os

In [2]:
Ex_dir = os.path.join(os.getcwd(), 'Ex14')
os.makedirs(os.path.join(Ex_dir, 'data'), exist_ok = True)
data_dir = os.path.join(Ex_dir, 'data')

In [28]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

'cuda'

In [3]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)) 
])
train_dataset = datasets.MNIST(root = data_dir, train = True, download = True, transform = transform)
test_dataset = datasets.MNIST(root = data_dir, train = False, transform = transform)
train_loader  = DataLoader(train_dataset, batch_size = 128)
test_loader = DataLoader(test_dataset, batch_size = 128)

In [39]:
# test thử đầu ra của lớp rnn torch
images = torch.randn(4,28,3*28)
rnn = nn.RNN(input_size = 3*28, hidden_size = 64, num_layers = 2, batch_first = True)
output = rnn(images)
output[0].shape, output[1].shape

(torch.Size([4, 28, 64]), torch.Size([2, 4, 64]))

input có dạng batch * input_size (số time step) * input_dim

Đầu ra của Rnn là 1 tuple có 2 tensor tensor đầu tiên là các trạng thái ẩn trước đó, tensor cuối cùng là trạng thái ẩn hiện tại 
* Nếu stack x hidden state thì chiều của tensor đầu vẫn giữ nguyên (nếu stack thì truyền đi từ h1 đến hx và chỉ lấy hx), tensor cuối có dạng x,batch,hidden_dim
* Phần tử cuối cùng của tensor đầu luôn giống phần từ cuối cùng của tensor 2 (đều là final hidden state hiện tại)

In [42]:
class RNNModel(nn.Module):
    # input dim là chiều của x, hidden_dim là chiều của h, layer_dim là số hidden state, output dim là chiều của kết quả dự báo cuối cùng
    def __init__(self, input_dim, hidden_dim, layer_dim, output_dim):
        super(RNNModel, self).__init__()
        self.hidden_dim = hidden_dim
        self.layer_dim = layer_dim
        self.rnn = nn.RNN(input_size = input_dim, hidden_size = hidden_dim, num_layers = layer_dim, batch_first=True, nonlinearity='relu', bidirectional=False)
        self.fc = nn.Linear(hidden_dim, output_dim) # shape linear a,b là b,a

    def forward(self, X):
        h0 = torch.zeros((self.layer_dim, X.size(0), self.hidden_dim)).to(device) # khởi tạo 0 cho các layer hidden_state đầu tiên
        out, hn = self.rnn(X, h0) # có thể không cần điền h0 lúc này sẽ mặc định là 0
        out = self.fc(out[:, -1, :]) # lấy ra lớp ẩn cuối cùng
        return out

In [43]:
model = RNNModel(input_dim = 28, hidden_dim = 100, layer_dim = 1, output_dim = 10)
model.to(device)

RNNModel(
  (rnn): RNN(28, 100, batch_first=True)
  (fc): Linear(in_features=100, out_features=10, bias=True)
)

In [53]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr = 1e-3, momentum = 0.9)

for epoch in range(3):
    model.train()
    running_loss = 0.0
    for i, (input, label) in enumerate(train_loader):
        with torch.no_grad():
            # input đang có dạng batch, c, h, w chuyển về b,c*h, w
            input = input.view(input.size(0), input.size(1) * input.size(2), input.size(3))
        input, label = input.to(device), label.to(device)
        optimizer.zero_grad()
        output = model(input)
        loss = criterion(output, label)
        loss.backward()
        optimizer.step()

        # in stats cuối epoch:
        running_loss += loss.item()
        if i == len(train_loader) - 1:
            print(f'[Epoch: {epoch}, final_batch: {len(train_loader)}] loss: {running_loss / len(train_loader)}')
            running_loss = 0.0
print('finish training')

[Epoch: 0, final_batch: 469] loss: 0.9926330296596738
[Epoch: 1, final_batch: 469] loss: 0.6721577976685343
[Epoch: 2, final_batch: 469] loss: 0.508043769929709
finish training


In [54]:
# test
model.eval()
with torch.no_grad():
    correct = 0
    total = 0
    for (input, label) in test_loader:
        input = input.view(input.size(0), input.size(1) * input.size(2), input.size(3)).to(device)
        label = label.to(device)
        output = model(input)
        pred = output.argmax(dim = 1)
        correct += pred.eq(label.view_as(pred)).sum().item()
        total += label.size(0)

print(f'accuracy: {100.0*correct/total}')


accuracy: 81.37
